<a href="https://colab.research.google.com/github/chadallen/insider_trading_detection/blob/main/Detect_insider_trading_on_prediction_markets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/chadallen/insider_trading_detection/blob/main/Detect_insider_trading_on_prediction_markets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Polymarket Insider Trading Detector

**Links:** [GitHub](https://github.com/chadallen/insider_trading_detection) · [Google Drive](https://drive.google.com/drive/folders/1rD3SXjFoLfCmlQOeftVj1LI-pbEu1o1u?usp=sharing) · [Dashboard](https://polymarket-dashboard-roan.vercel.app/)

---

## What this does

Detects suspicious trading patterns in resolved Polymarket prediction markets using two signals:

**1. Price-based scoring** (free — Polymarket public API)
- VPIN, time-weighted VPIN, resolution surprise, late move ratio → **Isolation Forest**

**2. Wallet-based scoring** (costs Dune credits)
- New wallet ratio, burst score, directional consensus, trade VPIN → **rule-based score**

**3. RF Classifier** — combines all features into `insider_trading_prob` (0–1)

---

## Run order

### ⚡ Quick: re-run classifier on saved data (0 credits, ~2 min)
```
Cell 0 → Cell 1 → Cell 2 → Cell 13
```

### 🔄 Refresh price scores (free, ~10 min)
```
Cell 0 → Cell 1 → Cell 2 → Cell 3 → Cell 4 → Cell 5 → Cell 5b → Cell 6
→ Cell 11 → Cell 12 → Cell 13 → Cell 14 → Cell 15
```

### 💳 Full refresh with wallet data (~8 credits, ~25 min)
```
All cells in order: Cell 0 through Cell 15
```

| Cell | Purpose | Credits |
|------|---------|--------|
| 0 | Config | 0 |
| **1** | **Installs + ALL shared functions** | 0 |
| 2 | Load Drive checkpoints | 0 |
| 3 | Fetch markets (Gamma API) | 0 |
| 4 | Fetch price histories (CLOB, ~5 min) | 0 |
| 5 | Compute price features | 0 |
| 5b | Real VPIN from Dune | ~1 |
| 6 | Isolation Forest scoring | 0 |
| 7 | Load labeled cases | 0 |
| 8 | Fetch labeled market trades [EXPENSIVE] | ~4 |
| 9 | Extract wallet features | 0 |
| 10 | Validate labeled markets | 0 |
| 11 | Top N wallet query [EXPENSIVE] | ~4 |
| 12 | Combine price + wallet scores | 0 |
| **13** | **RF Classifier → insider_trading_prob** | 0 |
| 14 | Save to Drive | 0 |
| 15 | Push CSVs to GitHub | 0 |


In [ ]:
#@title Cell 0 — User Configuration
# ╔══════════════════════════════════════════════════════════════════╗
# ║  EDIT THIS CELL BEFORE RUNNING ANYTHING ELSE                   ║
# ╚══════════════════════════════════════════════════════════════════╝

GITHUB_REPO   = "chadallen/insider_trading_detection"
GITHUB_BRANCH = "main"
DRIVE_FOLDER  = "insider_trading_poc"

# API keys — add as Colab Secrets (key icon in left sidebar):
#   DUNE_API_KEY  — from dune.com (free tier works)
#   GITHUB_TOKEN  — github.com/settings/tokens with repo scope

TOP_N_MARKETS = 50   # markets to run wallet query for in Cell 11

print("✅ Configuration loaded")
print(f"   Repo: {GITHUB_REPO} | Drive: {DRIVE_FOLDER} | Top N: {TOP_N_MARKETS}")


In [ ]:
#@title Cell 1 — Install dependencies, imports, and ALL shared functions
# ┌─────────────────────────────────────────────────────────────────────┐
# │  Run this cell FIRST every session.                                │
# │  All functions live here — after running Cell 1 you can jump to   │
# │  any other cell without running intermediate cells first.          │
# └─────────────────────────────────────────────────────────────────────┘

# ── Installs ─────────────────────────────────────────────────────────────
!pip install requests pandas numpy scikit-learn plotly PyGithub -q

# ── Imports ──────────────────────────────────────────────────────────────
import requests
import pandas as pd
import numpy as np
import json, os, pickle, time, io
from datetime import datetime, timedelta, timezone
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from github import Github

# ── API keys ──────────────────────────────────────────────────────────────
try:
    from google.colab import userdata
    DUNE_API_KEY = userdata.get("DUNE_API_KEY")
    GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
except Exception:
    DUNE_API_KEY = os.environ.get("DUNE_API_KEY", "")
    GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN", "")

DUNE_HEADERS = {"X-Dune-Api-Key": DUNE_API_KEY, "Content-Type": "application/json"}
_session_credits_used = 0.0

# ════════════════════════════════════════════════════════════════════════
# SECTION 1A — PRICE HISTORY FUNCTIONS
# ════════════════════════════════════════════════════════════════════════

def fetch_price_history(token_id, resolution_time, hours_before=48):
    """Fetch CLOB price history for a single market."""
    if isinstance(resolution_time, str):
        res_time = datetime.fromisoformat(resolution_time.replace("Z", "+00:00"))
    else:
        res_time = resolution_time
    start_time = res_time - timedelta(hours=hours_before)
    params = {
        "market": token_id, "interval": "max", "fidelity": 720,
        "startTs": int(start_time.timestamp()),
        "endTs":   int(res_time.timestamp()),
    }
    try:
        r = requests.get("https://clob.polymarket.com/prices-history", params=params, timeout=10)
        r.raise_for_status()
        data = r.json()
        if not data or "history" not in data or not data["history"]:
            return pd.DataFrame()
        df = pd.DataFrame(data["history"]).rename(columns={"t": "timestamp", "p": "price"})
        df["timestamp"] = pd.to_datetime(df["timestamp"], unit="s", utc=True)
        df["price"] = df["price"].astype(float)
        return df
    except Exception:
        return pd.DataFrame()

def compute_vpin_from_prices(history_df):
    if len(history_df) < 3: return None
    prices = history_df["price"].values
    changes = np.diff(prices)
    buy  = np.where(changes > 0,  changes, 0).sum()
    sell = np.where(changes < 0, -changes, 0).sum()
    total = buy + sell
    return abs(buy - sell) / total if total > 0 else 0.0

def compute_time_weighted_vpin(history_df):
    """VPIN with linear weights — moves closer to resolution count more."""
    if len(history_df) < 3: return None
    prices  = history_df["price"].values
    changes = np.diff(prices)
    weights = np.arange(1, len(changes) + 1, dtype=float)
    w_buy  = np.where(changes > 0,  changes * weights, 0).sum()
    w_sell = np.where(changes < 0, -changes * weights, 0).sum()
    total  = w_buy + w_sell
    return abs(w_buy - w_sell) / total if total > 0 else 0.0

def compute_price_features(history_df):
    if len(history_df) < 2: return None
    prices  = history_df["price"].values
    changes = np.abs(np.diff(prices))
    return {
        "price_volatility": changes.std() if len(changes) > 1 else 0,
        "max_single_move":  changes.max() if len(changes) > 0 else 0,
        "final_price":      prices[-1],
        "starting_price":   prices[0],
        "total_price_move": abs(prices[-1] - prices[0]),
    }

def compute_resolution_surprise(history_df):
    if len(history_df) < 2: return None, None
    prices     = history_df["price"].values
    actual     = 1.0 if prices[-1] > 0.5 else 0.0
    surprise   = abs(actual - prices[0])
    total_move = abs(prices[-1] - prices[0])
    final_step = abs(prices[-1] - prices[-2])
    late_move_ratio = (final_step / total_move) if total_move > 0.01 else 0.0
    return surprise, late_move_ratio

# ════════════════════════════════════════════════════════════════════════
# SECTION 1B — DUNE API FUNCTIONS
# ════════════════════════════════════════════════════════════════════════

def sql_quote(s):
    return "'" + s.replace("'", "''") + "'"

def execute_sql(sql):
    r = requests.post(
        "https://api.dune.com/api/v1/sql/execute",
        headers=DUNE_HEADERS, json={"sql": sql, "performance": "medium"}
    )
    r.raise_for_status()
    return r.json()["execution_id"]

def poll_until_done(execution_id, timeout=180, interval=5):
    start = time.time()
    while time.time() - start < timeout:
        r = requests.get(
            f"https://api.dune.com/api/v1/execution/{execution_id}/status",
            headers=DUNE_HEADERS
        )
        r.raise_for_status()
        state = r.json()["state"]
        if state == "QUERY_STATE_COMPLETED": return True
        if state in ("QUERY_STATE_FAILED", "QUERY_STATE_CANCELLED"):
            print(f"  Query failed: {state}"); return False
        print(f"  Status: {state} — waiting {interval}s...")
        time.sleep(interval)
    print("  Timed out."); return False

def fetch_results(execution_id):
    r = requests.get(
        f"https://api.dune.com/api/v1/execution/{execution_id}/results",
        headers=DUNE_HEADERS
    )
    r.raise_for_status()
    return pd.DataFrame(r.json().get("result", {}).get("rows", []))

def get_execution_cost(execution_id):
    r = requests.get(
        f"https://api.dune.com/api/v1/execution/{execution_id}/status",
        headers=DUNE_HEADERS
    )
    r.raise_for_status()
    return r.json().get("execution_cost_credits")

def run_dune_query(sql, label="query", timeout=180):
    """Execute a Dune SQL query end-to-end. Returns (df, execution_id)."""
    global _session_credits_used
    try:
        exec_id = execute_sql(sql)
        print(f"  Execution ID: {exec_id}")
        if poll_until_done(exec_id, timeout=timeout):
            df   = fetch_results(exec_id)
            cost = get_execution_cost(exec_id)
            if cost is not None:
                _session_credits_used += cost
            print(f"  💳 Cost: {cost:.4f} credits | Session total: {_session_credits_used:.4f}")
            return df, exec_id
        return pd.DataFrame(), exec_id
    except Exception as e:
        print(f"  ❌ run_dune_query({label}): {e}")
        return pd.DataFrame(), None

# ════════════════════════════════════════════════════════════════════════
# SECTION 1C — LABELED MARKET SQL BUILDER
# ════════════════════════════════════════════════════════════════════════

def build_labeled_sql(config):
    res_ts = config["resolution"].replace("T", " ")
    qf     = config["question_filter"]
    s, e   = config["start"], config["end"]
    return f"""
WITH trades AS (
    SELECT block_time, maker, price, amount, token_outcome_name
    FROM polymarket_polygon.market_trades
    WHERE {qf}
    AND block_time BETWEEN TIMESTAMP '{s}' AND TIMESTAMP '{e}'
),
market_stats AS (
    SELECT COUNT(*) AS trade_count,
           COUNT(DISTINCT maker) AS unique_wallets,
           SUM(amount) AS total_volume
    FROM trades
),
resolution_times AS (SELECT TIMESTAMP '{res_ts}' AS res_time),
new_wallets_12h AS (
    SELECT SUM(t.amount) AS new_wallet_volume_12h FROM trades t
    CROSS JOIN resolution_times r
    WHERE t.maker IN (
        SELECT maker FROM (
            SELECT maker, MIN(block_time) AS first_seen FROM trades GROUP BY maker
        ) fw WHERE fw.first_seen >= r.res_time - INTERVAL '12' HOUR
    )
),
new_wallets_6h AS (
    SELECT SUM(t.amount) AS new_wallet_volume_6h FROM trades t
    CROSS JOIN resolution_times r
    WHERE t.maker IN (
        SELECT maker FROM (
            SELECT maker, MIN(block_time) AS first_seen FROM trades GROUP BY maker
        ) fw WHERE fw.first_seen >= r.res_time - INTERVAL '6' HOUR
    )
),
burst AS (
    SELECT MAX(cnt) AS burst_score
    FROM (SELECT DATE_TRUNC('hour', block_time) AS h, COUNT(*) AS cnt
          FROM trades GROUP BY DATE_TRUNC('hour', block_time)) x
),
directional AS (
    SELECT MAX(ov) * 1.0 / NULLIF(SUM(ov), 0) AS directional_consensus
    FROM (SELECT token_outcome_name, SUM(amount) AS ov
          FROM trades GROUP BY token_outcome_name) x
),
vpin AS (
    SELECT ABS(yes_vol - no_vol) / NULLIF(yes_vol + no_vol, 0) AS trade_vpin
    FROM (SELECT SUM(CASE WHEN price > 0.5  THEN amount ELSE 0 END) AS yes_vol,
                 SUM(CASE WHEN price <= 0.5 THEN amount ELSE 0 END) AS no_vol
          FROM trades) x
)
SELECT ms.trade_count, ms.unique_wallets, ms.total_volume,
    COALESCE(nw12.new_wallet_volume_12h, 0) / NULLIF(ms.total_volume, 0) AS new_wallet_ratio_12h,
    COALESCE(nw6.new_wallet_volume_6h,  0) / NULLIF(ms.total_volume, 0) AS new_wallet_ratio_6h,
    b.burst_score, d.directional_consensus, v.trade_vpin
FROM market_stats ms
CROSS JOIN new_wallets_12h nw12 CROSS JOIN new_wallets_6h nw6
CROSS JOIN burst b CROSS JOIN directional d CROSS JOIN vpin v
"""

# ════════════════════════════════════════════════════════════════════════
# SECTION 1D — WALLET SUSPICION SCORING
# ════════════════════════════════════════════════════════════════════════

def compute_wallet_suspicion_score(features):
    """Rule-based wallet suspicion score 0.0–1.0. Each of 5 signals contributes up to 0.20."""
    score = 0.0
    nwr = features.get("new_wallet_ratio", 0)
    if nwr > 0.50:   score += 0.20
    elif nwr > 0.10: score += 0.10
    nwr_6h = features.get("new_wallet_ratio_6h", 0)
    if nwr_6h > 0.10:   score += 0.20
    elif nwr_6h > 0.05: score += 0.10
    vpin = features.get("trade_vpin", 0)
    if vpin > 0.80:   score += 0.20
    elif vpin > 0.60: score += 0.10
    burst = features.get("burst_score", 0)
    if burst > 200:  score += 0.20
    elif burst > 50: score += 0.10
    dc = features.get("directional_consensus", 0)
    if dc > 0.70:   score += 0.20
    elif dc > 0.55: score += 0.10
    return round(score, 2)

# ════════════════════════════════════════════════════════════════════════
# SECTION 1E — RF CLASSIFIER
# ════════════════════════════════════════════════════════════════════════

RF_FEATURES = [
    "vpin_score",
    "time_weighted_vpin",
    "surprise_score",
    "late_move_ratio",
    "price_volatility",
    "burst_score",
    "directional_consensus",
    "trade_vpin",
]

POSITIVE_KEYWORDS = [
    "Maduro", "Machado", "Khamenei", "Venezuela",
    "ZachXBT", "Taylor Swift", "Ophelia", "Gemini 3", "Iran",
]

def train_rf_classifier(df, features=None, pos_keywords=None, n_neg=30):
    """
    Train Random Forest on df_combined. Returns (df_with_probs, model, scaler).
    Positives: rows whose question matches POSITIVE_KEYWORDS (labeled insider trading cases).
    Negatives: bottom n_neg rows by combined_score (implicitly clean markets).
    Adds insider_trading_prob column (0-1) to df.
    """
    if features     is None: features     = RF_FEATURES
    if pos_keywords is None: pos_keywords = POSITIVE_KEYWORDS
    df = df.copy()

    # Label positives
    df["is_positive"] = df["question"].apply(
        lambda q: any(kw.lower() in str(q).lower() for kw in pos_keywords)
    )
    n_pos = df["is_positive"].sum()
    print(f"  Positives matched: {n_pos}")

    # Label negatives (bottom of combined_score = implicitly clean)
    n_neg = min(n_neg, len(df) - n_pos)
    neg_idx = (
        df[~df["is_positive"]]
        .dropna(subset=features)
        .nsmallest(n_neg, "combined_score")
        .index
    )
    df["is_negative"] = False
    df.loc[neg_idx, "is_negative"] = True
    print(f"  Negatives used:    {len(neg_idx)} (bottom by combined_score)")

    # Build training set
    df_train = df[df["is_positive"] | df["is_negative"]].dropna(subset=features).copy()
    X_train  = df_train[features].values
    y_train  = df_train["is_positive"].astype(int).values
    print(f"  Training set:      {y_train.sum()} pos | {(y_train==0).sum()} neg | {len(df_train)} total")

    if y_train.sum() == 0:
        print("  ❌ No positives found — check POSITIVE_KEYWORDS vs df_combined['question']")
        df["insider_trading_prob"] = np.nan
        return df, None, None

    # Train
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_train)
    rf = RandomForestClassifier(
        n_estimators=200, class_weight="balanced",
        max_depth=4, min_samples_leaf=2, random_state=42
    )
    rf.fit(X_scaled, y_train)

    # Score all markets that have complete features
    df_scoreable = df.dropna(subset=features).copy()
    X_all = scaler.transform(df_scoreable[features].values)
    df_scoreable["insider_trading_prob"] = rf.predict_proba(X_all)[:, 1]

    # Merge back on index (avoids question string mismatch issues)
    if "insider_trading_prob" in df.columns:
        df = df.drop(columns=["insider_trading_prob"])
    df = df.join(df_scoreable[["insider_trading_prob"]], how="left")

    return df, rf, scaler

print("✅ Cell 1 complete — all functions defined")
print("   Price:  fetch_price_history, compute_vpin_from_prices,")
print("           compute_time_weighted_vpin, compute_price_features, compute_resolution_surprise")
print("   Dune:   sql_quote, execute_sql, poll_until_done, fetch_results,")
print("           get_execution_cost, run_dune_query")
print("   Wallet: build_labeled_sql, compute_wallet_suspicion_score")
print("   ML:     train_rf_classifier  (uses RF_FEATURES, POSITIVE_KEYWORDS)")


In [ ]:
#@title Cell 2 — Mount Google Drive and load saved checkpoints
# Loads pkl files saved from a previous run. Variables that are None will
# be recomputed when their cell runs.
# ⏱ ~30 seconds · 0 credits

from google.colab import drive
try:
    drive.mount("/content/drive")
except ValueError:
    print("Drive already mounted ✅")

SAVE_DIR = f"/content/drive/MyDrive/{DRIVE_FOLDER}"
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Save directory: {SAVE_DIR}\n")

def load_checkpoint(name):
    path = f"{SAVE_DIR}/{name}.pkl"
    if os.path.exists(path):
        with open(path, "rb") as f: data = pickle.load(f)
        print(f"  ✅ Loaded {name}")
        return data
    print(f"  ⚠️  Not found: {name} — will be computed when its cell runs")
    return None

print("Loading checkpoints...")
df_markets    = load_checkpoint("df_markets")
df_scored     = load_checkpoint("df_scored")
histories     = load_checkpoint("histories")
dune_results  = load_checkpoint("dune_results")
df_wallet_agg = load_checkpoint("df_wallet_agg")
df_combined   = load_checkpoint("df_combined")

print("\nDone.")
print("If all ✅ above, you can jump to Cell 13 to re-run the classifier.")


---
## Section A: Price-Based Scoring (Cells 3–6)

Uses Polymarket's free public API. No credits required.

**Skip to Cell 6** if `df_scored` loaded from Drive ✅ — Isolation Forest can re-run on saved features.

> Cell 4 takes 3–5 minutes (170 markets × CLOB API).

In [ ]:
#@title Cell 3 — Fetch resolved political markets
# tag_id=2 (Politics), paginated, sorted by volume desc.
# Filters: volume ≥ $10M, end_date ≥ 2025-01-01.
# ⏱ ~30 seconds · 0 credits

POLITICS_TAG_ID  = 2
MARKETS_PER_PAGE = 100
MAX_PAGES        = 10

all_markets, seen_ids = [], set()
print(f"📡 Fetching closed political markets (tag_id={POLITICS_TAG_ID})...")

for page in range(MAX_PAGES):
    offset = page * MARKETS_PER_PAGE
    url = (
        f"https://gamma-api.polymarket.com/events"
        f"?tag_id={POLITICS_TAG_ID}&closed=true"
        f"&limit={MARKETS_PER_PAGE}&offset={offset}"
        f"&order=volume&ascending=false"
    )
    events = requests.get(url, timeout=30).json()
    if not events:
        print(f"  ✅ No more results at offset {offset}"); break

    new_count = 0
    for event in events:
        for mkt in event.get("markets", []):
            mid = mkt.get("conditionId")
            if not mid or mid in seen_ids: continue
            seen_ids.add(mid)
            try:
                token_ids = json.loads(mkt.get("clobTokenIds", "[]"))
                token_id  = token_ids[0] if token_ids else None
            except Exception:
                token_id = None
            all_markets.append({
                "market_id":       mid,
                "question":        mkt.get("question", ""),
                "end_date":        mkt.get("endDate", ""),
                "volume":          float(mkt.get("volume") or 0),
                "resolution_time": mkt.get("endDate"),
                "token_id":        token_id,
                "category":        "politics",
                "event_title":     event.get("title", ""),
            })
            new_count += 1

    earliest = min((m["end_date"] for m in all_markets), default="?")
    print(f"  Page {page+1}: +{new_count} | Total: {len(all_markets)} | Earliest: {earliest[:10]}")
    time.sleep(0.3)

df_markets = pd.DataFrame(all_markets)
df_markets = df_markets[df_markets["volume"]   >= 10_000_000].reset_index(drop=True)
df_markets = df_markets[df_markets["end_date"] >= "2025-01-01"].reset_index(drop=True)

print(f"\n✅ {len(df_markets)} markets | volume ≥ $10M | end_date ≥ 2025-01-01")
print(f"   Date range: {df_markets['end_date'].min()[:10]} → {df_markets['end_date'].max()[:10]}")
print(df_markets[["question","volume","end_date"]].sort_values("volume", ascending=False).head(10))


In [ ]:
#@title Cell 4 — Fetch price histories
# Calls CLOB API for each market. Skips markets with starting price outside 0.15–0.85
# (uncontested markets have no meaningful price signal).
# ⏱ 3–5 minutes · 0 credits

print(f"Fetching price histories for {len(df_markets)} markets...")
histories = {}

for i, (_, row) in enumerate(df_markets.iterrows()):
    if i % 25 == 0: print(f"  {i}/{len(df_markets)}...")
    if row["resolution_time"] is None: continue
    history = fetch_price_history(row["token_id"], row["resolution_time"])
    if len(history) < 3: continue
    if not (0.15 <= history["price"].iloc[0] <= 0.85): continue
    histories[row["token_id"]] = history

print(f"\n✅ Price histories cached for {len(histories)}/{len(df_markets)} markets")


In [ ]:
#@title Cell 5 — Compute price features
# VPIN, time-weighted VPIN, resolution surprise, late move ratio, volatility.
# ⏱ ~5 seconds · 0 credits

results = []
for _, row in df_markets.iterrows():
    history = histories.get(row["token_id"])
    if history is None or len(history) < 3: continue
    vpin            = compute_vpin_from_prices(history)
    tw_vpin         = compute_time_weighted_vpin(history)
    feats           = compute_price_features(history)
    surprise, late_move = compute_resolution_surprise(history)
    if any(v is None for v in [vpin, tw_vpin, feats, surprise, late_move]): continue
    results.append({
        "question":           row["question"],
        "volume":             row["volume"],
        "vpin_score":         vpin,
        "time_weighted_vpin": tw_vpin,
        "surprise_score":     surprise,
        "late_move_ratio":    late_move,
        **feats,
    })

df_scored = pd.DataFrame(results)
print(f"✅ Computed features for {len(df_scored)} markets")
print(df_scored[["question","vpin_score","surprise_score","late_move_ratio"]].head(5).to_string())


In [ ]:
#@title Cell 5b — Replace price-proxy VPIN with real volume-weighted VPIN from Dune [~1 credit]
# Overwrites vpin_score with real on-chain data. Drops markets with no Dune match.
# Requires: Cell 1 (run_dune_query, sql_quote), Cell 5 (df_scored)
# ⏱ ~2–4 min · ~1 credit

in_clause = ",\n    ".join(sql_quote(q) for q in df_scored["question"].tolist())

vpin_sql = f"""
WITH trades AS (
    SELECT question, price, amount
    FROM polymarket_polygon.market_trades
    WHERE question IN (
    {in_clause}
)
),
vpin AS (
    SELECT question,
        SUM(CASE WHEN price > 0.5  THEN amount ELSE 0 END) AS yes_vol,
        SUM(CASE WHEN price <= 0.5 THEN amount ELSE 0 END) AS no_vol,
        SUM(amount) AS total_vol
    FROM trades GROUP BY question
)
SELECT question,
       ABS(yes_vol - no_vol) / NULLIF(total_vol, 0) AS trade_vpin,
       total_vol
FROM vpin
"""

print(f"── Submitting VPIN query for {len(df_scored)} markets ──────────────")
df_vpin, _ = run_dune_query(vpin_sql, label="vpin_all_markets", timeout=300)

if not df_vpin.empty:
    vpin_lookup = dict(zip(df_vpin["question"], df_vpin["trade_vpin"].astype(float)))
    before = len(df_scored)
    df_scored = df_scored[df_scored["question"].isin(vpin_lookup)].copy()
    df_scored["vpin_score"] = df_scored["question"].map(vpin_lookup)
    print(f"  ✅ {len(df_scored)} markets with real VPIN | {before - len(df_scored)} dropped (no Dune data)")
    print(df_scored["vpin_score"].describe().round(4).to_string())
else:
    print("  ⚠️ Dune returned no results — df_scored unchanged")

print("\nRun Cell 6 to re-score with Isolation Forest using updated VPIN.")


In [ ]:
#@title Cell 6 — Score markets with Isolation Forest
# Unsupervised anomaly detection. contamination=0.1 → top ~10% flagged as anomalous.
# Adds suspicion_score to df_scored. Can re-run on loaded data without re-fetching.
# ⏱ ~2 seconds · 0 credits

feature_cols = [
    "vpin_score", "time_weighted_vpin", "volume",
    "total_price_move", "price_volatility", "max_single_move",
    "surprise_score", "late_move_ratio",
]

X        = df_scored[feature_cols].fillna(0)
X_scaled = StandardScaler().fit_transform(X)
iso = IsolationForest(contamination=0.1, random_state=42)
df_scored["anomaly_score"]   = iso.fit_predict(X_scaled)
df_scored["suspicion_score"] = -iso.decision_function(X_scaled)

top = df_scored.nlargest(15, "suspicion_score")[
    ["question", "suspicion_score", "vpin_score", "surprise_score", "late_move_ratio", "volume"]
].reset_index(drop=True)
top.index += 1
print(f"✅ Scored {len(df_scored)} markets\n")
print("Top 15 by price suspicion score:")
print(top.to_string())


---
## Section B: Wallet-Based Scoring (Cells 7–12)

Queries on-chain trade data from `polymarket_polygon.market_trades` via Dune Analytics.

| Cell | What it does | Credits | When to run |
|------|-------------|---------|-------------|
| 7 | Load labeled ground truth cases | 0 | Always |
| 8 | Fetch Dune trades for labeled markets | ~4 | Only when adding new labeled cases |
| 9 | Extract wallet features from results | 0 | After Cell 8 |
| 10 | Validate scoring against ground truth | 0 | After Cell 9 |
| 11 | Top N wallet query | ~4 | When top markets change |
| 12 | Combine price + wallet → df_combined | 0 | After Cell 11 |

**If `dune_results`, `df_wallet_agg`, `df_combined` loaded from Drive ✅:** Skip to Cell 13.

In [ ]:
#@title Cell 7 — Load labeled insider trading cases
# Ground truth for validation and classifier training.
# Labels: CONFIRMED = credibly reported · SUSPECTED = anomalous/unconfirmed · POSSIBLE = uninvestigated
# ⏱ Instant · 0 credits

LABELS_CSV = """market_question,label
"Will the US strike Iran by February 28, 2026?",CONFIRMED
"Will María Corina Machado win the Nobel Peace Prize in 2025?",CONFIRMED
"Will Nicolás Maduro be removed from office by January 31, 2026?",SUSPECTED
"US x Venezuela military engagement by January 15, 2026?",SUSPECTED
"Which crypto company will ZachXBT expose for insider trading?",SUSPECTED
"Khamenei out as Supreme Leader of Iran by January 31?",SUSPECTED
"Taylor Swift x Travis Kelce get married by December 31?",SUSPECTED
"Will The Fate of Ophelia by Taylor Swift be the #1 searched song on Google this year?",SUSPECTED
"Gemini 3.0 Flash released by December 31?",POSSIBLE
"""

df_labels = pd.read_csv(io.StringIO(LABELS_CSV))
print(f"✅ Loaded {len(df_labels)} labeled cases")
print(df_labels[["market_question", "label"]].to_string())


In [ ]:
#@title Cell 8 — Fetch labeled market trades from Dune [~4 credits]
# One aggregated query per labeled market. Skip if dune_results loaded from Drive.
# ⏱ ~5 min · ~4 credits

LABELED_MARKET_CONFIGS = {
    "iran":               {"label": "CONFIRMED",  "question_filter": "question = 'US strikes Iran by February 28, 2026?'",                                                           "start": "2026-02-26", "end": "2026-02-28", "resolution": "2026-02-28T23:59:00"},
    "nobel":              {"label": "CONFIRMED",  "question_filter": "question = 'Will María Corina Machado win the Nobel Peace Prize in 2025?'",                                    "start": "2025-10-08", "end": "2025-10-11", "resolution": "2025-10-10T11:00:00"},
    "maduro":             {"label": "SUSPECTED",  "question_filter": "question LIKE '%Maduro%'",                                                                                     "start": "2026-01-28", "end": "2026-01-31", "resolution": "2026-01-31T00:00:00"},
    "venezuela_military": {"label": "SUSPECTED",  "question_filter": "question = 'US x Venezuela military engagement by January 15, 2026?'",                                         "start": "2026-01-01", "end": "2026-01-04", "resolution": "2026-01-03T00:00:00"},
    "zachxbt":            {"label": "SUSPECTED",  "question_filter": "question = 'Will Axiom be accused of insider trading?'",                                                       "start": "2026-02-24", "end": "2026-02-27", "resolution": "2026-02-26T23:59:00"},
    "khamenei":           {"label": "SUSPECTED",  "question_filter": "question = 'Khamenei out as Supreme Leader of Iran by January 31?'",                                           "start": "2026-01-28", "end": "2026-02-01", "resolution": "2026-01-31T00:00:00"},
    "taylor_wedding":     {"label": "SUSPECTED",  "question_filter": "question = 'Taylor Swift x Travis Kelce get married by December 31?'",                                         "start": "2025-12-28", "end": "2026-01-01", "resolution": "2025-12-31T23:59:00"},
    "alpha_raccoon":      {"label": "SUSPECTED",  "question_filter": "question = 'Will The Fate of Ophelia by Taylor Swift be the #1 searched song on Google this year?'",           "start": "2025-11-28", "end": "2025-12-05", "resolution": "2025-12-04T00:00:00"},
    "gemini":             {"label": "POSSIBLE",   "question_filter": "question = 'Gemini 3.0 Flash released by December 31?'",                                                      "start": "2025-12-10", "end": "2025-12-18", "resolution": "2025-12-17T00:00:00"},
}

dune_results = {}
for i, (name, config) in enumerate(LABELED_MARKET_CONFIGS.items(), 1):
    print(f"\n── [{i}/{len(LABELED_MARKET_CONFIGS)}] {name.upper()} ({config['label']}) ────────────")
    df, _ = run_dune_query(build_labeled_sql(config), label=name)
    dune_results[name] = df
    if not df.empty:
        row = df.iloc[0]
        print(f"  ✅ trades={int(row.get('trade_count',0)):,} wallets={int(row.get('unique_wallets',0)):,} vol=${float(row.get('total_volume',0)):,.0f}")
    else:
        print("  ⚠️ No results")

print(f"\n✅ Fetched {sum(1 for d in dune_results.values() if not d.empty)}/{len(dune_results)} markets")
print("Run Cell 14 to save dune_results to Drive.")


In [ ]:
#@title Cell 9 — Extract wallet features from Dune results
# Reads aggregated results from Cell 8 (or Drive). Flags low-volume markets.
# ⏱ Instant · 0 credits

MARKET_LABELS  = {k: v["label"] for k, v in LABELED_MARKET_CONFIGS.items()}
COLUMN_MAP = {
    "new_wallet_ratio_12h":  "new_wallet_ratio",
    "new_wallet_ratio_6h":   "new_wallet_ratio_6h",
    "trade_vpin":            "trade_vpin",
    "burst_score":           "burst_score",
    "directional_consensus": "directional_consensus",
    "total_volume":          "total_volume",
    "unique_wallets":        "unique_wallets",
    "trade_count":           "trade_count",
}
MIN_VOLUME_USD = 1000

wallet_features = {}
for name, label in MARKET_LABELS.items():
    df = dune_results.get(name)
    if df is None or len(df) == 0:
        print(f"⚠️  {name}: no data — re-run Cell 8"); continue
    row   = df.iloc[0]
    feats = {col: row.get(src, 0) for src, col in COLUMN_MAP.items()}
    wallet_features[name] = feats
    flag  = " ⚠️ LOW VOLUME" if feats.get("total_volume", 0) < MIN_VOLUME_USD else ""
    print(f"── {name.upper()} ({label}){flag}")
    print(f"   Vol ${feats['total_volume']:>14,.0f} | Wallets {int(feats['unique_wallets']):,}")
    print(f"   New wallet 6h {feats['new_wallet_ratio_6h']:.1%} | VPIN {feats['trade_vpin']:.3f} | Burst {int(feats['burst_score'])} | Dir {feats['directional_consensus']:.1%}\n")

print(f"✅ Extracted features for {len(wallet_features)}/{len(MARKET_LABELS)} markets")


In [ ]:
#@title Cell 10 — Validate wallet scoring on labeled markets
# Applies compute_wallet_suspicion_score to all ground truth cases.
# Requires: wallet_features (Cell 9)
# ⏱ Instant · 0 credits

print(f"{"Market":<22} {"Label":<11} {"Score":>6}  Dominant signal")
print("─" * 68)

for name, label in MARKET_LABELS.items():
    feats = wallet_features.get(name)
    if feats is None:
        print(f"{name:<22} {label:<11}    N/A  (no data)"); continue
    score    = compute_wallet_suspicion_score(feats)
    dominant = max([
        ("new_wallet_12h",  feats.get("new_wallet_ratio",     0)),
        ("new_wallet_6h",   feats.get("new_wallet_ratio_6h",  0)),
        ("trade_vpin",      feats.get("trade_vpin",           0)),
        ("burst",           feats.get("burst_score",          0) / 1000),
        ("dir_consensus",   feats.get("directional_consensus",0)),
    ], key=lambda x: x[1])
    flag = " ⚠️ LOW VOL" if feats.get("total_volume", 0) < 1000 else ""
    print(f"{name:<22} {label:<11} {score:>6.2f}  {dominant[0]}={dominant[1]:.3f}{flag}")


---
## Section C: Top-N Wallet Query & Combine (Cells 11–12)

**Cell 11** runs the aggregated Dune query for the top `TOP_N_MARKETS` from df_scored.
**Cell 12** merges price + wallet scores into df_combined.

**If `df_wallet_agg` and `df_combined` loaded from Drive ✅:** Skip to Cell 13.

In [ ]:
#@title Cell 11 — Top N wallet query from Dune [~4 credits]
# Fetches burst_score, directional_consensus, trade_vpin for top N markets.
# Re-run when top markets change after a price refresh.
# ⏱ ~3–5 min · ~4 credits

top_markets = df_scored.nlargest(TOP_N_MARKETS, "suspicion_score")
in_clause   = ",\n    ".join(sql_quote(q) for q in top_markets["question"].tolist())

wallet_sql = f"""
WITH trades AS (
    SELECT question, maker, price, amount, token_outcome_name, block_time
    FROM polymarket_polygon.market_trades
    WHERE question IN (
    {in_clause}
)
    AND block_time >= NOW() - INTERVAL '90' DAY
),
burst AS (
    SELECT question, MAX(cnt) AS burst_score
    FROM (SELECT question, DATE_TRUNC('hour', block_time) AS h, COUNT(*) AS cnt
          FROM trades GROUP BY question, DATE_TRUNC('hour', block_time)) x
    GROUP BY question
),
directional AS (
    SELECT question,
           MAX(ov) * 1.0 / NULLIF(SUM(ov), 0) AS directional_consensus
    FROM (SELECT question, token_outcome_name, SUM(amount) AS ov
          FROM trades GROUP BY question, token_outcome_name) x
    GROUP BY question
),
vpin AS (
    SELECT question,
           ABS(SUM(CASE WHEN price > 0.5  THEN amount ELSE 0 END) -
               SUM(CASE WHEN price <= 0.5 THEN amount ELSE 0 END)) /
           NULLIF(SUM(amount), 0) AS trade_vpin,
           SUM(amount)    AS total_volume,
           COUNT(*)       AS trade_count,
           COUNT(DISTINCT maker) AS unique_wallets
    FROM trades GROUP BY question
)
SELECT v.question, v.trade_vpin, v.total_volume, v.trade_count,
       v.unique_wallets, b.burst_score, d.directional_consensus
FROM vpin v
JOIN burst b       ON v.question = b.question
JOIN directional d ON v.question = d.question
"""

print(f"── Wallet query for top {TOP_N_MARKETS} markets ────────────────────────")
df_wallet_agg, _ = run_dune_query(wallet_sql, label="top_n_wallet", timeout=300)

if not df_wallet_agg.empty:
    print(f"\n✅ {len(df_wallet_agg)} markets returned")
    print(df_wallet_agg[["question","burst_score","directional_consensus","trade_vpin"]].head(10).to_string())
else:
    print("⚠️ No results — check query or Dune status")


In [ ]:
#@title Cell 12 — Combine price + wallet scores into df_combined
# Merges df_scored with df_wallet_agg on question.
# Normalizes suspicion_score to 0-1 as price_score.
# Computes wallet_score via rules, then averages with price_score.
# Also merges raw feature columns needed by the RF classifier (Cell 13).
# ⏱ Instant · 0 credits

# ── Price score: normalize suspicion_score to 0–1 ────────────────────────
df_price = df_scored[[
    "question", "suspicion_score",
    "vpin_score", "time_weighted_vpin",
    "surprise_score", "late_move_ratio", "price_volatility"
]].copy()
s = df_price["suspicion_score"]
df_price["price_score"] = (s - s.min()) / (s.max() - s.min() + 1e-9)

# ── Wallet score via rules ────────────────────────────────────────────────
df_wallet = df_wallet_agg.copy() if df_wallet_agg is not None and not df_wallet_agg.empty else pd.DataFrame()
if not df_wallet.empty:
    df_wallet["wallet_score"] = df_wallet.apply(
        lambda row: compute_wallet_suspicion_score(row.to_dict()), axis=1
    )

# ── Merge ─────────────────────────────────────────────────────────────────
wallet_cols = ["question", "wallet_score", "burst_score", "directional_consensus", "trade_vpin"]
df_combined = df_price.merge(
    df_wallet[wallet_cols] if not df_wallet.empty and all(c in df_wallet.columns for c in wallet_cols)
    else pd.DataFrame(columns=["question"]),
    on="question", how="left"
)

df_combined["combined_score"] = df_combined.apply(
    lambda r: (
        (r["price_score"] + r["wallet_score"]) / 2
        if pd.notna(r.get("wallet_score")) else r["price_score"]
    ), axis=1
)

print(f"✅ df_combined: {len(df_combined)} markets")
print(f"   With wallet score: {df_combined['wallet_score'].notna().sum() if 'wallet_score' in df_combined.columns else 0}")
print("\n── Top 15 by combined_score ─────────────────────────────────────────")
top = df_combined.nlargest(15, "combined_score")[
    ["question", "price_score", "wallet_score", "combined_score"]
].reset_index(drop=True)
top.index += 1
print(top.to_string())


---
## Section D: RF Classifier & Output (Cells 13–15)

**Minimum run order to re-run classifier on saved data:**
```
Cell 0 → Cell 1 → Cell 2 → Cell 13
```

> `train_rf_classifier()` is defined in Cell 1. It requires `df_combined` with `combined_score`, `price_score`, and raw feature columns. If loaded from Drive after Cell 12 was previously run, all columns should be present.

In [ ]:
#@title Cell 13 — Random Forest classifier → insider_trading_prob
# Trains RF on labeled positives + implicit negatives.
# Adds insider_trading_prob (0–1) as the primary ranking signal.
#
# Requires: Cell 0 + Cell 1 + df_combined (from Cell 2 or Cell 12)
#
# To tune: edit POSITIVE_KEYWORDS or RF_FEATURES in Cell 1.
# ⏱ ~5 seconds · 0 credits

print("── Training RF classifier ───────────────────────────────────────────")
df_combined, rf_model, rf_scaler = train_rf_classifier(df_combined)

if rf_model is None:
    print("\n⚠️  Model not trained — check diagnostic above.")
else:
    # Feature importances
    print("\n🌲 Feature importances:")
    for feat, imp in sorted(zip(RF_FEATURES, rf_model.feature_importances_), key=lambda x: -x[1]):
        bar = "█" * int(imp * 40)
        print(f"  {feat:<25} {bar} {imp:.3f}")

    # Top 15
    print("\n── Top 15 by insider_trading_prob ──────────────────────────────────")
    top = (
        df_combined[df_combined["insider_trading_prob"].notna()]
        .nlargest(15, "insider_trading_prob")
        [["question", "insider_trading_prob", "combined_score"]]
        .reset_index(drop=True)
    )
    top.index += 1
    print(top.to_string())

    # Sanity check — where did labeled cases land?
    print("\n── Labeled cases (sanity check) ────────────────────────────────────")
    labeled = df_combined[df_combined.get("is_positive", pd.Series(False, index=df_combined.index)) == True][
        ["question", "insider_trading_prob", "combined_score"]
    ].sort_values("insider_trading_prob", ascending=False)
    if labeled.empty:
        print("  (none matched — check POSITIVE_KEYWORDS)")
    else:
        print(labeled.to_string())

    n_scored = df_combined["insider_trading_prob"].notna().sum()
    print(f"\n✅ Scored {n_scored}/{len(df_combined)} markets")
    print(f"   {len(df_combined) - n_scored} skipped (missing features — need wallet data from Cell 11)")
    print("\nRun Cell 14 to save, Cell 15 to push to GitHub.")


In [ ]:
#@title Cell 14 — Save all data to Google Drive
# Persists pkl checkpoints. Next session can skip expensive re-fetches.
# ⏱ ~30 seconds · 0 credits

def save_checkpoint(name, data):
    if data is None:
        print(f"  ⚠️  Skipping {name} (None)"); return
    path = f"{SAVE_DIR}/{name}.pkl"
    with open(path, "wb") as f: pickle.dump(data, f)
    print(f"  ✅ Saved {name}")

print("Saving to Drive...")
save_checkpoint("df_markets",    df_markets)
save_checkpoint("df_scored",     df_scored)
save_checkpoint("histories",     histories)
save_checkpoint("dune_results",  dune_results)
save_checkpoint("df_wallet_agg", df_wallet_agg)
save_checkpoint("df_combined",   df_combined)
print("\n✅ Done.")


In [ ]:
#@title Cell 15 — Push results to GitHub
# Pushes df_combined, df_scored, df_wallet_agg as CSVs to outputs/ folder.
# Dashboard at https://polymarket-dashboard-roan.vercel.app/ reads these live.
# ⏱ ~30 seconds · 0 credits

g    = Github(GITHUB_TOKEN)
repo = g.get_repo(GITHUB_REPO)

def push_csv(df, filename, message):
    if df is None or (hasattr(df, 'empty') and df.empty):
        print(f"  ⚠️  Skipping {filename} (empty)"); return
    content = df.to_csv(index=False)
    path    = f"outputs/{filename}"
    try:
        existing = repo.get_contents(path, ref=GITHUB_BRANCH)
        repo.update_file(path, message, content, existing.sha, branch=GITHUB_BRANCH)
        print(f"  ✅ Updated  {path}")
    except Exception:
        repo.create_file(path, message, content, branch=GITHUB_BRANCH)
        print(f"  ✅ Created  {path}")

ts = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")
print(f"Pushing to {GITHUB_REPO} ({ts})...")
push_csv(df_combined,   "df_combined.csv",   f"Update combined scores {ts}")
push_csv(df_scored,     "df_scored.csv",     f"Update price scores {ts}")
push_csv(df_wallet_agg, "df_wallet_agg.csv", f"Update wallet scores {ts}")
print("\n✅ Done — dashboard will update within ~1 minute.")
